# 82_linear_recurrence

Audience: junior researcher

- Challenge path: `challenges/medium/82_linear_recurrence`
- Source spec: [challenges/medium/82_linear_recurrence/challenge.html](../challenge.html)
- Source implementation: [challenges/medium/82_linear_recurrence/challenge.py](../challenge.py)


## Problem Statement

The original challenge HTML is embedded below so the notebook stays close to the repo source.

<p>
  Given two matrices <code>a</code> and <code>x</code>, each of shape <code>[B, L]</code> (batch size &times; sequence length),
  compute the linear recurrence <code>h</code> of shape <code>[B, L]</code> defined by:
  <code>h[b, 0] = x[b, 0]</code> and <code>h[b, t] = a[b, t] &times; h[b, t&minus;1] + x[b, t]</code> for <code>t &ge; 1</code>.
  All values are <code>float32</code>. This operation is the core computational primitive of
  State Space Models (SSMs) such as Mamba, S4, and H3.
</p>

<svg width="640" height="200" viewBox="0 0 640 200" xmlns="http://www.w3.org/2000/svg"
     style="display:block; margin:20px auto;">
  <rect width="640" height="200" fill="#222" rx="8"/>
  <!-- Title -->
  <text x="320" y="24" fill="#ccc" font-size="13" text-anchor="middle" font-family="monospace">Linear Recurrence: h[t] = a[t] · h[t-1] + x[t]</text>
  <!-- Boxes for h values -->
  <rect x="40"  y="80" width="80" height="40" rx="4" fill="#1a3a5c" stroke="#4a9eff" stroke-width="1.5"/>
  <rect x="180" y="80" width="80" height="40" rx="4" fill="#1a3a5c" stroke="#4a9eff" stroke-width="1.5"/>
  <rect x="320" y="80" width="80" height="40" rx="4" fill="#1a3a5c" stroke="#4a9eff" stroke-width="1.5"/>
  <rect x="460" y="80" width="80" height="40" rx="4" fill="#1a3a5c" stroke="#4a9eff" stroke-width="1.5"/>
  <text x="80"  y="105" fill="#4a9eff" font-size="13" text-anchor="middle" font-family="monospace">h[0]</text>
  <text x="220" y="105" fill="#4a9eff" font-size="13" text-anchor="middle" font-family="monospace">h[1]</text>
  <text x="360" y="105" fill="#4a9eff" font-size="13" text-anchor="middle" font-family="monospace">h[2]</text>
  <text x="500" y="105" fill="#4a9eff" font-size="13" text-anchor="middle" font-family="monospace">h[3]</text>
  <!-- Arrows between h values with a[t] labels -->
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
      <path d="M0,0 L0,6 L8,3 Z" fill="#7ec8a0"/>
    </marker>
  </defs>
  <line x1="120" y1="100" x2="176" y2="100" stroke="#7ec8a0" stroke-width="1.5" marker-end="url(#arr)"/>
  <text x="148" y="94" fill="#7ec8a0" font-size="11" text-anchor="middle" font-family="monospace">×a[1]</text>
  <line x1="260" y1="100" x2="316" y2="100" stroke="#7ec8a0" stroke-width="1.5" marker-end="url(#arr)"/>
  <text x="288" y="94" fill="#7ec8a0" font-size="11" text-anchor="middle" font-family="monospace">×a[2]</text>
  <line x1="400" y1="100" x2="456" y2="100" stroke="#7ec8a0" stroke-width="1.5" marker-end="url(#arr)"/>
  <text x="428" y="94" fill="#7ec8a0" font-size="11" text-anchor="middle" font-family="monospace">×a[3]</text>
  <!-- x[t] inputs from below -->
  <line x1="80"  y1="152" x2="80"  y2="124" stroke="#ccc" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="220" y1="152" x2="220" y2="124" stroke="#ccc" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="360" y1="152" x2="360" y2="124" stroke="#ccc" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="500" y1="152" x2="500" y2="124" stroke="#ccc" stroke-width="1.2" marker-end="url(#arr)"/>
  <text x="80"  y="170" fill="#ccc" font-size="12" text-anchor="middle" font-family="monospace">x[0]</text>
  <text x="220" y="170" fill="#ccc" font-size="12" text-anchor="middle" font-family="monospace">x[1]</text>
  <text x="360" y="170" fill="#ccc" font-size="12" text-anchor="middle" font-family="monospace">x[2]</text>
  <text x="500" y="170" fill="#ccc" font-size="12" text-anchor="middle" font-family="monospace">x[3]</text>
  <!-- Plus signs -->
  <text x="80"  y="147" fill="#ccc" font-size="13" text-anchor="middle" font-family="monospace">+</text>
  <text x="220" y="147" fill="#ccc" font-size="13" text-anchor="middle" font-family="monospace">+</text>
  <text x="360" y="147" fill="#ccc" font-size="13" text-anchor="middle" font-family="monospace">+</text>
  <text x="500" y="147" fill="#ccc" font-size="13" text-anchor="middle" font-family="monospace">+</text>
  <!-- Ellipsis -->
  <text x="590" y="105" fill="#ccc" font-size="18" text-anchor="middle" font-family="monospace">…</text>
</svg>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The result must be stored in the output tensor <code>h</code></li>
</ul>

<h2>Examples</h2>

<p>Example 1 — exponential decay (<code>a = 0.5</code>, single impulse):</p>
\[
a = \begin{bmatrix} 0.5 & 0.5 & 0.5 & 0.5 \end{bmatrix}, \quad
x = \begin{bmatrix} 1.0 & 0.0 & 0.0 & 0.0 \end{bmatrix}
\]
\[
h = \begin{bmatrix} 1.0 & 0.5 & 0.25 & 0.125 \end{bmatrix}
\]

<p>Example 2 — prefix sum (<code>a = 1</code>, unit inputs):</p>
\[
a = \begin{bmatrix} 1.0 & 1.0 & 1.0 & 1.0 \end{bmatrix}, \quad
x = \begin{bmatrix} 1.0 & 1.0 & 1.0 & 1.0 \end{bmatrix}
\]
\[
h = \begin{bmatrix} 1.0 & 2.0 & 3.0 & 4.0 \end{bmatrix}
\]

<p>Full example with <code>B = 2</code>, <code>L = 4</code>:</p>
\[
a = \begin{bmatrix} 0.5 & 0.5 & 0.5 & 0.5 \\ 1.0 & 1.0 & 1.0 & 1.0 \end{bmatrix}, \quad
x = \begin{bmatrix} 1.0 & 0.0 & 0.0 & 0.0 \\ 1.0 & 1.0 & 1.0 & 1.0 \end{bmatrix}
\]
\[
h = \begin{bmatrix} 1.0 & 0.5 & 0.25 & 0.125 \\ 1.0 & 2.0 & 3.0 & 4.0 \end{bmatrix}
\]

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>B</code> &le; 256 (batch size)</li>
  <li>1 &le; <code>L</code> &le; 65,536 (sequence length)</li>
  <li>All values in <code>a</code> and <code>x</code> are <code>float32</code></li>
  <li>Performance is measured with <code>B</code> = 64, <code>L</code> = 16,384</li>
</ul>


## Framework Coverage

This notebook collects the currently available solution artifacts for PyTorch, JAX, Triton, and MLX.

## Pytorch

No solution file is present yet.

## Jax

No solution file is present yet.

## Triton

No solution file is present yet.

## Mlx

No solution file is present yet.

## Verification Notes

Use `python scripts/verify_matrix_solutions.py` for the local matrix-operation verifier.
GPU-only Triton validation still depends on a remote NVIDIA environment.